# 02 - Preprocessing Pipeline

This notebook demonstrates the complete preprocessing pipeline for the Adversarial Robust IDS.

**Sections:**
1. Load raw data
2. Cleaning (missing values, infinities, duplicates)
3. Label encoding (binary and multiclass)
4. Categorical feature encoding (one-hot)
5. Feature scaling (StandardScaler)
6. Feature selection (Mutual Information)
7. SMOTE class balancing
8. Save / load artifacts

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from src.preprocessing.data_loader import DataLoader, CATEGORY_MAP
from src.preprocessing.preprocessor import DataPreprocessor
from src.preprocessing.feature_engineering import compute_feature_importance, get_correlation_matrix
from src.utils.config import load_config

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('Imports loaded.')

## 1. Load Raw Data

In [ ]:
config = load_config('../config/config.yaml')
loader = DataLoader()

try:
    df = loader.load(config)
except FileNotFoundError:
    print('Dataset not found -- using synthetic data.')
    df = loader.generate_synthetic(n_samples=5000)

print(f'Raw data shape: {df.shape}')
df.head(3)

## 2. Cleaning -- Before / After

The `DataPreprocessor.clean_data()` method handles:
- Replace infinities with NaN
- Fill NaN using the configured strategy (median / mean / drop)
- Remove duplicate rows

In [ ]:
preprocessor = DataPreprocessor(config)

print('--- Before cleaning ---')
print(f'Shape       : {df.shape}')
print(f'Missing vals: {df.isnull().sum().sum()}')
print(f'Duplicates  : {df.duplicated().sum()}')

df_clean = preprocessor.clean_data(df)

print()
print('--- After cleaning ---')
print(f'Shape       : {df_clean.shape}')
print(f'Missing vals: {df_clean.isnull().sum().sum()}')
print(f'Duplicates  : {df_clean.duplicated().sum()}')

## 3. Label Encoding

Two label schemes are created:
- **Binary** -- Normal (0) vs Attack (1)
- **Multiclass** -- Normal=0, DoS=1, Probe=2, R2L=3, U2R=4

In [ ]:
df_encoded = preprocessor.encode_labels(df_clean)

print('Category mapping:', CATEGORY_MAP)
print()
print('Binary label distribution:')
print(df_encoded['label_binary'].value_counts())
print()
print('Multiclass label distribution:')
print(df_encoded['label_multiclass'].value_counts().sort_index())

## 4. Categorical Feature Encoding (One-Hot)

Categorical columns (`protocol_type`, `service`, `flag`) are one-hot encoded.

In [ ]:
cat_cols_before = df_encoded.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns before encoding: {cat_cols_before}')
print(f'Shape before encoding: {df_encoded.shape}')

df_ohe = preprocessor.encode_categorical(df_encoded)
cat_cols_after = df_ohe.select_dtypes(include='object').columns.tolist()

print(f'\nShape after one-hot encoding: {df_ohe.shape}')
print(f'Remaining object columns (labels only): {cat_cols_after}')
print(f'New binary columns added: {df_ohe.shape[1] - df_encoded.shape[1]}')

## 5. Run Full Pipeline

The `run_pipeline()` method chains all steps: clean -> encode labels -> encode categoricals -> split -> scale -> feature select -> SMOTE.

In [ ]:
preprocessor2 = DataPreprocessor(config)
data = preprocessor2.run_pipeline(df, label_type='multiclass')

print('Pipeline outputs:')
for key in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test']:
    print(f'  {key:8s} shape: {data[key].shape}  dtype: {data[key].dtype}')

print(f'\n  n_features : {data["n_features"]}')
print(f'  n_classes  : {data["n_classes"]}')
print(f'  label_type : {data["label_type"]}')

## 6. Feature Scaling Verification

After StandardScaler, the training data should have approximately zero mean and unit variance.

In [ ]:
train_means = data['X_train'].mean(axis=0)
train_stds = data['X_train'].std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(train_means, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title(f'Feature Means (should be near 0)\nActual mean of means: {train_means.mean():.4f}')
axes[0].axvline(0, color='red', linestyle='--')

axes[1].hist(train_stds, bins=30, color='coral', edgecolor='white')
axes[1].set_title(f'Feature Stds (should be near 1)\nActual mean of stds: {train_stds.mean():.4f}')
axes[1].axvline(1, color='red', linestyle='--')

plt.tight_layout()
plt.show()

## 7. Feature Selection Results

Top features are selected using mutual information between each feature and the target label.

In [ ]:
selected = data['feature_names']
print(f'Selected {len(selected)} features (out of original):')
for i, name in enumerate(selected, 1):
    print(f'  {i:2d}. {name}')

In [ ]:
# Feature importance scores from the selector
if preprocessor2.feature_selector is not None:
    scores = preprocessor2.feature_selector.scores_
    mask = preprocessor2.feature_selector.get_support()
    selected_scores = scores[mask]

    fig, ax = plt.subplots(figsize=(12, 6))
    sorted_idx = np.argsort(selected_scores)
    ax.barh(range(len(sorted_idx)), selected_scores[sorted_idx], color='teal')
    ax.set_yticks(range(len(sorted_idx)))
    ax.set_yticklabels([selected[i] for i in sorted_idx], fontsize=8)
    ax.set_xlabel('Mutual Information Score')
    ax.set_title('Selected Feature Importances')
    plt.tight_layout()
    plt.show()

## 8. SMOTE Balance Verification

After SMOTE, all classes in the training set should be roughly balanced.

In [ ]:
from collections import Counter

class_dist = Counter(data['y_train'])
class_names = {v: k for k, v in CATEGORY_MAP.items()}

print('Training set class distribution after SMOTE:')
for cls_id in sorted(class_dist.keys()):
    name = class_names.get(cls_id, f'Class {cls_id}')
    print(f'  {name:8s} (id={cls_id}): {class_dist[cls_id]:6d}')

fig, ax = plt.subplots(figsize=(8, 4))
labels = [class_names.get(k, f'Class {k}') for k in sorted(class_dist.keys())]
counts = [class_dist[k] for k in sorted(class_dist.keys())]
ax.bar(labels, counts, color=['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6'][:len(labels)])
ax.set_title('Training Set Distribution After SMOTE')
ax.set_ylabel('Count')
for i, v in enumerate(counts):
    ax.text(i, v + 10, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 9. Saved Artifacts

The pipeline saves `scaler.pkl`, `label_encoder.pkl`, and `feature_selector.pkl` for inference-time reuse.

In [ ]:
from src.utils.config import resolve_path

artifact_dir = resolve_path('models/preprocessing')
print(f'Artifact directory: {artifact_dir}')

for fname in ['scaler.pkl', 'label_encoder.pkl', 'feature_selector.pkl']:
    fpath = os.path.join(artifact_dir, fname)
    exists = os.path.exists(fpath)
    size = os.path.getsize(fpath) if exists else 0
    print(f'  {fname:25s}  exists={exists}  size={size} bytes')

In [ ]:
# Demonstrate loading artifacts and transforming new data
preprocessor3 = DataPreprocessor(config)
preprocessor3.load_artifacts()

print('Loaded scaler type    :', type(preprocessor3.scaler).__name__)
print('Loaded selector type  :', type(preprocessor3.feature_selector).__name__)
print('\nReady for inference-time transformation.')

## Summary

The preprocessing pipeline:
1. Cleans missing / infinite values and removes duplicates
2. Creates binary and multiclass label encodings
3. One-hot encodes categorical features
4. Splits data 70/15/15 with stratification
5. Applies StandardScaler (fit on train only)
6. Selects top-k features via mutual information
7. Balances training classes with SMOTE
8. Persists artifacts for later inference

Next: `03_tier2_training.ipynb` -- train DNN, CNN, LSTM models.